# RSNA Knee — Kaggle runner

Thin wrapper. All logic lives in the repo's `src/`, which arrives here as an
attached Kaggle Dataset imported from GitHub.

**Order matters:** refresh the dataset on Kaggle *before* re-running, or this
executes the previous revision. Cell 2 prints the revision so a stale run is
obvious instead of silent.

Settings: Accelerator `GPU T4 x2`, and Internet `off` for submission.
Note T4 (not P100) — Kaggle's PyTorch ships without Pascal kernels.

In [ ]:
# Locate the code dataset. Kaggle may nest the repo one level down inside the
# dataset root, so probe both levels. Deliberately shallow: a recursive walk
# would crawl every DICOM in the competition data.
import os

CODE = None
for entry in sorted(os.listdir("/kaggle/input")):
    base = f"/kaggle/input/{entry}"
    if not os.path.isdir(base):
        continue
    candidates = [base] + [f"{base}/{x}" for x in sorted(os.listdir(base))]
    for path in candidates:
        if os.path.exists(f"{path}/src/train.py"):
            CODE = path

print("CODE =", CODE)
assert CODE, "no dataset under /kaggle/input contains src/train.py"

In [ ]:
# Staleness check: does this match the VERSION you just pushed?
print(open(f"{CODE}/VERSION").read())

In [ ]:
# Debug run first. Prints the real data layout, then walks a handful of
STUDIES through the full pipeline and writes submission.csv.
import runpy, sys

sys.argv = ["train.py", "--config", f"{CODE}/configs/base.yaml"]
runpy.run_path(f"{CODE}/src/train.py", run_name="__main__")

## After the debug run is clean

1. Set `debug: false` in `configs/base.yaml` (or pass `--no-debug`).
2. Bump `VERSION`, push, refresh the Kaggle dataset.
3. Confirm the printed marker changed, then run full.

Full runs need checkpointing — a session is capped at ~12h and gets killed
at the limit, so never leave the only copy of a trained model in memory.